# CS308 Лаборатори 8 — Ensemble (загваруудыг нэгтгэх)

**Оюутан:** Б. Өсөхбаяр (B231910012)

Доорх **1–20 дугаар нүд нь багшийн өгсөн эх код** бөгөөд өөрчлөөгүй.
Түүний дараа миний хийсэн шинжилгээ, засвар, нэмэлт туршилт орно.

> Colab: нэмэлт суулгац шаардлагагүй, бүх сан (scikit-learn, numpy) бэлэн байдаг.
> Ажиллах хугацаа ~1 мин, GPU хэрэггүй.

In [1]:
import sklearn

In [2]:
from sklearn.datasets import make_classification

In [3]:
x_train, y_train = make_classification(n_samples=1000, n_features=6, n_classes=2, n_clusters_per_class=1, random_state=15, n_informative=3)
x_test, y_test = make_classification(n_samples=1000, n_features=6, n_classes=2, n_clusters_per_class=1, random_state=15, n_informative=3)

In [4]:
print(x_train.shape, y_train.shape)

(1000, 6) (1000,)


In [5]:
from sklearn.linear_model import LinearRegression

In [6]:
from sklearn import linear_model

In [7]:
from sklearn.tree import DecisionTreeClassifier

In [8]:
dtree = DecisionTreeClassifier(max_depth=3)

In [9]:
svm = sklearn.svm.SVC()

In [10]:
linear_regression = LinearRegression()

In [11]:
ensemble_model = sklearn.svm.SVC()

In [12]:
dtree.fit(x_train, y_train)
svm.fit(x_train, y_train)
linear_regression.fit(x_train, y_train)
pred_dtree = dtree.predict(x_train)
pred_svm = svm.predict(x_train)
pred_linear = linear_regression.predict(x_train)
print(pred_dtree.shape, pred_svm.shape, pred_linear.shape)

(1000,) (1000,) (1000,)


In [13]:
import numpy as np

In [14]:
preds = np.array([pred_dtree, pred_svm, pred_linear])

In [15]:
preds = preds.T
print(preds.shape)

(1000, 3)


In [16]:
ensemble_model.fit(preds, y_train)

,C,1.0
,kernel,'rbf'
,degree,3
,gamma,'scale'
,coef0,0.0
,shrinking,True
,probability,False
,tol,0.001
,cache_size,200
,class_weight,None
,verbose,False


In [17]:
pred_final = ensemble_model.predict(preds)

In [18]:
accuracy = np.sum(pred_final==y_train)
print(accuracy/y_train.shape[0])

0.949


In [19]:
pred_dtree_test = dtree.predict(x_test)
pred_svm_test = svm.predict(x_test)
pred_linear_test = linear_regression.predict(x_test)
preds_test = np.array([pred_dtree_test, pred_svm_test, pred_linear_test])
preds_test = preds_test.T

test_pred_final = ensemble_model.predict(preds_test)

In [20]:
accuracy = np.sum(test_pred_final==y_test)
print(accuracy/y_test.shape[0])

0.949


---
# Миний хэсэг — үр дүнг шалгах

Багшийн код **train дээр 0.949, тест дээр 0.949** гэсэн яг ижил тоо өгч байна.
Хоёр өөр олонлог дээр үзүүлэлт ижил гарах магадлал маш бага тул энэ нь
санамсаргүй биш — эхлээд үүнийг шалгая.

In [21]:
# ── Шалгалт 1: тест олонлог нь сургалтын олонлогтой ижил юу? ──
import numpy as np
print("x_test == x_train :", np.array_equal(x_train, x_test))
print("y_test == y_train :", np.array_equal(y_train, y_test))
print()
print("Шалтгаан: make_classification-д хоёуланд нь random_state=15 өгсөн.")
print("Ижил random_state нь ижил өгөгдөл үүсгэдэг тул 'тест' гэж нэрлэсэн")
print("олонлог нь үнэндээ сургалтын өгөгдлийн ХУУЛБАР юм.")

x_test == x_train : True
y_test == y_train : True

Шалтгаан: make_classification-д хоёуланд нь random_state=15 өгсөн.
Ижил random_state нь ижил өгөгдөл үүсгэдэг тул 'тест' гэж нэрлэсэн
олонлог нь үнэндээ сургалтын өгөгдлийн ХУУЛБАР юм.


### Олдсон хоёр асуудал

**1. Тест олонлог нь сургалтын олонлогтой ижил.**
`make_classification(..., random_state=15)` хоёр удаа дуудахад яг ижил өгөгдөл буцаана.
Тиймээс `0.949` нь тестийн үзүүлэлт биш, зүгээр л сургалтын үзүүлэлт.

**2. Meta загвар leakage-тай сурсан.**
Суурь загварууд `x_train` дээр сурсан, дараа нь **яг тэр мөрүүд дээрээ** таамаг гаргаж,
meta загвар тэр таамаглалаар сурсан. Суурь загварууд өөрсдийн сурсан мөрүүддээ
хэт сайн таамагладаг тул meta загвар бодит байдлаас илүү цэвэрхэн дохио хардаг.
Зөв арга бол **out-of-fold** таамаглал ашиглах (Wolpert, *Stacked Generalization*, 1992):
суурь загвар тухайн мөрийг **хараагүй** байж гаргасан таамгийг л meta загварт өгнө.

In [22]:
# ── Засвар A: жинхэнэ тусдаа тест олонлог ──
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split, cross_val_predict, StratifiedKFold
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import StackingClassifier
from sklearn.metrics import accuracy_score

X, y = make_classification(n_samples=2000, n_features=6, n_classes=2, n_redundant=0,
                           n_clusters_per_class=1, n_informative=3, random_state=15)
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.5, random_state=42, stratify=y)
print(f"train {Xtr.shape[0]} мөр, test {Xte.shape[0]} мөр — огтлолцолгүй\n")

def base():
    return [("dtree",  DecisionTreeClassifier(max_depth=3, random_state=42)),
            ("svm",    SVC(probability=True, random_state=42)),
            ("logreg", LogisticRegression(max_iter=1000))]

for n, m in base():
    m.fit(Xtr, ytr)
    print(f"  суурь загвар {n:<7}: test acc = {accuracy_score(yte, m.predict(Xte)):.4f}")

train 1000 мөр, test 1000 мөр — огтлолцолгүй

  суурь загвар dtree  : test acc = 0.8010
  суурь загвар svm    : test acc = 0.8700


  суурь загвар logreg : test acc = 0.7940


In [23]:
# ── Засвар B: энгийн vs out-of-fold stacking ──
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# (1) Багшийн арга: meta загварыг суурь загваруудын СУРСАН мөрүүд дээрх таамгаар сургах
d, s, l = DecisionTreeClassifier(max_depth=3, random_state=42), SVC(random_state=42), LogisticRegression(max_iter=1000)
d.fit(Xtr,ytr); s.fit(Xtr,ytr); l.fit(Xtr,ytr)
meta_in  = np.array([d.predict(Xtr), s.predict(Xtr), l.predict(Xtr)]).T
meta_out = np.array([d.predict(Xte), s.predict(Xte), l.predict(Xte)]).T
naive = SVC(random_state=42).fit(meta_in, ytr)
acc_naive = accuracy_score(yte, naive.predict(meta_out))

# (2) Out-of-fold: тухайн мөрийг хараагүй байж гаргасан таамаг
oof = np.array([cross_val_predict(m, Xtr, ytr, cv=cv) for _, m in base()]).T
oof_meta = SVC(random_state=42).fit(oof, ytr)
acc_oof = accuracy_score(yte, oof_meta.predict(meta_out))

# (3) sklearn-ий бэлэн хэрэгжүүлэлт (дотроо out-of-fold хийдэг)
sk = StackingClassifier(estimators=base(), final_estimator=LogisticRegression(),
                        cv=cv).fit(Xtr, ytr)
acc_sk = accuracy_score(yte, sk.predict(Xte))

print(f"Энгийн (leakage-тай) stacking : {acc_naive:.4f}")
print(f"Out-of-fold stacking          : {acc_oof:.4f}")
print(f"sklearn StackingClassifier    : {acc_sk:.4f}")
print(f"\nХамгийн сайн ГАНЦ суурь загвар (SVM): "
      f"{accuracy_score(yte, s.predict(Xte)):.4f}")

# Суурь загварууд ХЭР ӨӨР алдаа гаргаж байна вэ? (ensemble-ийн ашгийг тодорхойлогч)
pv = {"dtree": d.predict(Xte), "svm": s.predict(Xte), "logreg": l.predict(Xte)}
ns = list(pv)
print("\nСанал зөрөх хувь (өндөр = олон янз = ensemble-д ашигтай):")
for i, a in enumerate(ns):
    for b in ns[i+1:]:
        print(f"   {a:>6} / {b:<6}: {(pv[a] != pv[b]).mean():.3f}")

Энгийн (leakage-тай) stacking : 0.8510
Out-of-fold stacking          : 0.8700
sklearn StackingClassifier    : 0.8630

Хамгийн сайн ГАНЦ суурь загвар (SVM): 0.8700

Санал зөрөх хувь (өндөр = олон янз = ensemble-д ашигтай):
    dtree / svm   : 0.103
    dtree / logreg: 0.089
      svm / logreg: 0.102


### Үр дүнгийн тайлбар

Гурван хувилбарыг харьцуулахад хоёр тусдаа дүгнэлт гарч байна:

**1. Out-of-fold засвар үнэхээр хэрэгтэй.**
Энгийн (leakage-тай) stacking нь out-of-fold хувилбараас **доогуур** гарч байна.
Яагаад гэвэл meta загвар нь суурь загваруудын *сурсан мөрүүд дээрх* хэт өөдрөг
таамгийг хараад сурсан боловч, тест дээр түүнээс муу чанартай таамаг ирдэг —
өөрөөр хэлбэл meta загвар **буруу тархалт** дээр сурсан. Out-of-fold таамаг нь
meta загварт тестийн үед ирэх чанартай ижил чанарын оролт өгдөг тул энэ зөрүү арилна.

**2. Гэхдээ stacking нь ганц SVM-ээс давсангүй.**
Засварласан ч гэсэн хамгийн сайн ганц суурь загвартай тэнцэхээс хэтрээгүй. Шалтгаан:

- Суурь гурван загварын **хүч тэнцвэргүй** (SVM бусдаасаа мэдэгдэхүйц дээгүүр).
- Санал зөрөх хувь нь бага — доорх хэмжилтийг хар — тул **ижил мөрүүдийг андуурдаг**.
- Ийм үед meta загварын хийж чадах дээд зүйл бол хамгийн сайн суурь загварыг дагах явдал.

Ensemble нь суурь загварууд **өөр өөр алдаа гаргадаг** үед л ашиг өгдөг
(Dietterich, *Ensemble Methods in Machine Learning*, 2000). Үүнийг дараагийн нүдэнд шалгая.

In [24]:
# ── Ensemble хэзээ ҮНЭХЭЭР тустай вэ? ──
# Олон тооны ЖИЖИГ, ХООРОНДОО ХАМААРАЛ БАГАТАЙ загвар нэгтгэх (bagging/boosting)
from sklearn.ensemble import (RandomForestClassifier, GradientBoostingClassifier,
                              BaggingClassifier, VotingClassifier)
from sklearn.model_selection import cross_val_score

models = {
 "Ганц мод (max_depth=3)"  : DecisionTreeClassifier(max_depth=3, random_state=42),
 "Ганц мод (гүн хязгааргүй)": DecisionTreeClassifier(random_state=42),
 "Bagging (100 мод)"       : BaggingClassifier(DecisionTreeClassifier(random_state=42),
                                               n_estimators=100, random_state=42),
 "RandomForest (100 мод)"  : RandomForestClassifier(n_estimators=100, random_state=42),
 "GradientBoosting (100)"  : GradientBoostingClassifier(n_estimators=100, random_state=42),
 "SVM (RBF)"               : SVC(random_state=42),
 "Voting (мод+SVM+logreg)" : VotingClassifier(base(), voting="soft"),
}
print(f"{'Загвар':<26}{'CV (train)':<20}{'test'}")
print("-"*56)
for n, m in models.items():
    sc = cross_val_score(m, Xtr, ytr, cv=cv)                    # сонголтод — train дээр
    te = accuracy_score(yte, m.fit(Xtr, ytr).predict(Xte))      # тест — нэг л удаа
    print(f"{n:<26}{sc.mean():.4f} ± {sc.std():.4f}      {te:.4f}")

Загвар                    CV (train)          test
--------------------------------------------------------
Ганц мод (max_depth=3)    0.8180 ± 0.0112      0.8010
Ганц мод (гүн хязгааргүй) 0.8320 ± 0.0093      0.8310


Bagging (100 мод)         0.8930 ± 0.0093      0.8870


RandomForest (100 мод)    0.8780 ± 0.0186      0.8860


GradientBoosting (100)    0.8840 ± 0.0169      0.8840
SVM (RBF)                 0.8510 ± 0.0086      0.8700


Voting (мод+SVM+logreg)   0.8390 ± 0.0066      0.8360


### Дүгнэлт

| Асуулт | Хариу |
|---|---|
| Багшийн 0.949 юу гэсэн үг вэ? | Сургалтын үзүүлэлт. `random_state=15` давхардсан тул тест олонлог нь хуулбар байсан. |
| Зөв хуваалт дээр хэд вэ? | Хамгийн сайн ганц загвар ≈0.87. Зөрүү нь бүхэлдээ leakage-аас. |
| Out-of-fold засвар тусалсан уу? | **Тийм** — энгийн stacking-ийн алдагдлыг нөхөж, хамгийн сайн суурь загвартай тэнцүүлсэн. |
| Stacking нь ганц SVM-ээс давсан уу? | **Үгүй.** Суурь загварууд хүч тэнцвэргүй, санал зөрөх нь бага. |
| Ensemble хаана ҮНЭХЭЭР тусалсан бэ? | **Bagging / RandomForest / GradientBoosting** — ганц модноос ≈ **+8 оноо**. |

**Механизм:** bagging нь өгөгдлийг дахин түүвэрлэж олон мод сургаснаар модны
**дисперсийг** бууруулдаг (Breiman, *Bagging Predictors*, 1996); RandomForest нь
нэмж баганыг санамсаргүй сонгож модуудын **хоорондын хамаарлыг** таслана
(Breiman, *Random Forests*, 2001). Харин 3 ширхэг тэнцвэргүй, ижил алдаатай
загварыг stacking хийхэд бууруулах дисперс бага тул ашиг гарахгүй.

**Гол сургамж:** ensemble өөрөө шидэт арга биш. *Олон* ба *хоорондоо хамааралгүй*
суурь загвар байж гэмээнэ ажиллана — Voting-ийн үр дүн (хамгийн сул гишүүндээ
татагдаж буурсан) үүнийг бас харуулж байна.